# 關鍵詞辨識 (第 2 版)：篩選詞彙 + CTC + 詞的時間位置

以 `sign_keyword.ipynb` 為基礎，做兩項改進：

| 改動 | 目的 |
|---|---|
| **篩選詞彙**：用上一版在**驗證集**的 AP，只保留學得起來的詞 (`SELECT_MIN_VAL_AP`，最多 `SELECT_MAX` 個) | 上一版 200 個詞裡有很多 AP 接近 0 (導致、透過、是否…)，留著只會干擾訓練與展示 |
| **CTC 輔助損失**：encoder 每一格也預測「這裡是哪個關鍵詞」，依字幕中關鍵詞的**出現順序**訓練 | 讓模型學出詞**大概在哪幾格**，而不只是「整句裡有沒有」；也多給 encoder 一份訓練訊號 |

`.npy` **不會被拆開**：輸入仍然是 Phase5 切好的整句，CTC 只是讓模型自己學出每個詞落在哪幾格。

**一次跑兩個版本** (同樣的篩選後詞彙)，才能分清楚進步是來自「篩選詞彙」還是「CTC」：

| 版本 | 內容 |
|---|---|
| `無CTC` | 只篩選詞彙，模型跟上一版相同 |
| `CTC` | 篩選詞彙 + CTC 輔助損失 |

臉部特徵在上一版沒有幫助，這一版預設只用手 (`USE_FACE = False`)；資料一樣讀 `03segments_face`，臉部特徵會被遮掉。

**最後會看到**：上一版 / 無CTC / CTC 的 mAP 比較、CTC 的效果是否顯著、每個詞的 AP，以及**模型認為每個詞出現在第幾秒**的範例圖。

**執行前**：要先跑過 `sign_keyword.ipynb` (需要它的 `keyword/keywords.json` 與 `keyword/<版本>/scores.npz` 來篩選詞彙)。

In [ ]:
import copy
import csv
import json
import math
import random
import re
import urllib.request
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

warnings.filterwarnings("ignore")
# os.chdir(r"C:\你的專案資料夾\Data Training")

# ================================================================ 路徑設定
SEG_TAG = "_face"          # 分段資料：03segments_face / 03labels_face (292 維)
LABEL_DIR = Path(r"..\Data Preparation\Phase5-Pause\output\03labels" + SEG_TAG)
SEG_DIR = Path(r"..\Data Preparation\Phase5-Pause\output\03segments" + SEG_TAG)
DATA_DIR = Path(r"data")                     # 沿用原本的資料切分 (句子跟其他 notebook 一一對應)
OUT_DIR = Path(r"keyword_ctc")               # 這一版的輸出，底下每個版本一個資料夾
PREV_DIR = Path(r"keyword")                  # 上一版 sign_keyword.ipynb 的輸出 (用來篩選詞彙)
JIEBA_DICT = DATA_DIR / "dict.txt.big"       # jieba 繁體詞典，找不到會自動下載
JIEBA_DICT_URL = "https://raw.githubusercontent.com/fxsjy/jieba/master/extra_dict/dict.txt.big"

# ================================================================ 關鍵詞設定
N_KEYWORDS = 200           # 取訓練集最常見的幾個詞
MIN_WORD_LEN = 2           # 至少幾個字 (單字太模糊，例如「的」「是」)
STOPWORDS = set()          # 想排除的詞，例如 {"表示", "已經"}
MIN_TEST_POSITIVE = 3      # 測試集至少出現幾次的詞才算進 mAP (太少的話分數不穩定)

# ================================================================ 篩選詞彙 (用上一版在「驗證集」的 AP，測試集完全不參與)
SELECT_MIN_VAL_AP = 0.1    # 驗證集 AP 至少多少才保留
SELECT_MAX = 100           # 最多保留幾個詞
PREV_VARIANTS = ["手+臉", "只有手"]   # 用上一版哪些版本的驗證集 AP (取平均)

# ================================================================ 這一版的設定
USE_FACE = False           # 上一版臉部特徵沒有幫助，預設只用手
VARIANTS = {               # 名稱: CTC 損失的權重 (0 = 不用 CTC)
    "無CTC": 0.0,
    "CTC": 0.5,
}
FPS = 29.97                # 範例圖換算秒數用 (01fps 裡大多是 29.97)

# ================================================================ 資料 (跟 face notebook 相同)
SPLIT = (0.8, 0.1, 0.1)
SEED = 42
SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]
MIN_FREQ = 2
FEAT_DIM = 292
FLAG_DIMS = [126, 127, 155, 291]
USE_BLENDSHAPES = USE_LIPS = USE_HEAD_POSE = USE_FACE   # 跟著 USE_FACE，不用手動改
FRAME_STRIDE = 2
MAX_FRAMES = 300
MAX_TEXT_LEN = 64
SPEED_RANGE = (0.8, 1.2)
NOISE_STD = 0.05
NORM_FILE = "norm_masked_face.npz"
AUGMENT = True
ROTATE_DEG = 10
SCALE_RANGE = (0.9, 1.1)
SHIFT = 0.05
TIME_MASK_N = 2
TIME_MASK_MAX = 0.1

# ================================================================ 模型與訓練
D_MODEL = 256
N_HEADS = 4
ENC_LAYERS = 4
FF_DIM = 1024
DROPOUT = 0.3
BATCH_SIZE = 32
EPOCHS = 150
LR = 3e-4
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 5
PATIENCE = 10              # val mAP 連續幾個 epoch 沒進步就停止
MAX_POS_WEIGHT = 10.0      # 少見的詞提高正例權重的上限

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
PAD, BOS, EOS, UNK = 0, 1, 2, 3
print("device:", DEVICE)

## DATASET
跟上一版相同。

In [ ]:
if all((DATA_DIR / name).exists() for name in ("train.csv", "val.csv", "test.csv", "vocab.json", "norm.npz")):
    print(f"沿用 {DATA_DIR} 裡現有的資料切分、字典與 norm")
else:
    print(f"{DATA_DIR} 沒有完整資料，重新建立")
    groups = {}
    for txt in sorted(LABEL_DIR.glob("*/*/*.txt")):
        group = txt.parent.relative_to(LABEL_DIR)
        for line in txt.read_text(encoding="utf-8").splitlines():
            name, _, text = line.partition(": ")
            if text.strip():
                groups.setdefault(group.as_posix(), []).append(((group / f"{name}.npy").as_posix(), text.strip()))

    keys = sorted(groups)
    random.Random(SEED).shuffle(keys)
    n_train, n_val = round(len(keys) * SPLIT[0]), round(len(keys) * SPLIT[1])
    splits = {"train": keys[:n_train], "val": keys[n_train:n_train + n_val], "test": keys[n_train + n_val:]}

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    rows = {}
    for split, split_keys in splits.items():
        rows[split] = [row for key in sorted(split_keys) for row in groups[key]]
        with (DATA_DIR / f"{split}.csv").open("w", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)
            writer.writerow(("path", "text"))
            writer.writerows(rows[split])

    freq = Counter(c for _, text in rows["train"] for c in text)
    chars = sorted(c for c, n in freq.items() if n >= MIN_FREQ)
    (DATA_DIR / "vocab.json").write_text(json.dumps(SPECIALS + chars, ensure_ascii=False), encoding="utf-8")

    total, total_sq, count = np.zeros(FEAT_DIM), np.zeros(FEAT_DIM), 0
    for path, _ in tqdm(rows["train"], desc="norm"):
        arr = np.load(SEG_DIR / path).astype(np.float64)
        total += arr.sum(0)
        total_sq += (arr ** 2).sum(0)
        count += len(arr)
    mean = total / count
    std = np.sqrt(np.maximum(total_sq / count - mean ** 2, 0)) + 1e-6
    mean[FLAG_DIMS], std[FLAG_DIMS] = 0.0, 1.0
    np.savez(DATA_DIR / "norm.npz", mean=mean.astype(np.float32), std=std.astype(np.float32))

for split in ("train", "val", "test"):
    with (DATA_DIR / f"{split}.csv").open(encoding="utf-8-sig", newline="") as f:
        print(f"{split}: {sum(1 for _ in csv.DictReader(f))} 句")
print("字元數:", len(json.loads((DATA_DIR / "vocab.json").read_text(encoding="utf-8"))) - len(SPECIALS))

# ---------------------------------------------------------------- 特徵位置與缺漏遮罩
LEFT, RIGHT, POSE = slice(0, 63), slice(63, 126), slice(128, 155)
LEFT_FLAG, RIGHT_FLAG, POSE_FLAG = 126, 127, 155
BLENDSHAPES, LIPS, HEAD_POSE, FACE_FLAG = slice(156, 208), slice(208, 288), slice(288, 291), 291


def valid_mask(arr):
    """(T, FEAT_DIM) 的 bool：每一維是不是真的有偵測到。旗標本身永遠算有效。"""
    m = np.ones(arr.shape, dtype=bool)
    m[:, LEFT] = arr[:, [LEFT_FLAG]] > 0.5
    m[:, RIGHT] = arr[:, [RIGHT_FLAG]] > 0.5
    point_ok = (arr[:, POSE].reshape(-1, 9, 3)[:, :, 2] > 0) & (arr[:, [POSE_FLAG]] > 0.5)   # 信心值 0 = 這個點沒偵測到
    m[:, POSE] = np.repeat(point_ok, 3, axis=1)
    m[:, 156:FACE_FLAG] = arr[:, [FACE_FLAG]] > 0.5            # 沒偵測到臉 → 臉部特徵全部無效
    for part, use in ((BLENDSHAPES, USE_BLENDSHAPES), (LIPS, USE_LIPS), (HEAD_POSE, USE_HEAD_POSE)):
        if not use:
            m[:, part] = False                                   # 關掉的特徵組一律設成 0
    if not (USE_BLENDSHAPES or USE_LIPS or USE_HEAD_POSE):
        m[:, FACE_FLAG] = False                                  # 完全不用臉時，連「有沒有臉」也不給
    return m


# ---------------------------------------------------------------- 先確認每一句的分段檔都存在
missing = Counter()
for split in ("train", "val", "test"):
    with (DATA_DIR / f"{split}.csv").open(encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            if not (SEG_DIR / row["path"]).exists():
                missing[row["path"].split("/")[0]] += 1
if missing:
    print(f"[錯誤] {SEG_DIR} 缺少 {sum(missing.values())} 個分段檔，各 part 缺少的數量：")
    for part, n in sorted(missing.items(), key=lambda x: (len(x[0]), x[0])):
        print(f"  part {part}：{n} 個")
    print("請依序確認缺少的 part：")
    print("  1. Phase4-Keypoint/Keypoint_output/01face/<part> 有沒有檔案 → 沒有的話跑 02keypoint_face.py")
    print("  2. Phase4-Keypoint/Keypoint_output/02combined_face/<part> 有沒有檔案 → 沒有的話跑 03keypoint_combine_face.py")
    print("  3. 03split_npy.py 設 FEATURE_TAG = \"_face\"、MARGIN = 0，part 輸入 all 重跑")
    print("  也檢查 Phase4-Keypoint/、Phase5-Pause/ 裡有沒有 failed_*_part_<part>.txt 失敗紀錄")
    raise FileNotFoundError(f"{SEG_DIR} 缺少分段檔，見上方說明")
print(f"{SEG_DIR} 的分段檔都齊全")

if (DATA_DIR / NORM_FILE).exists():
    print(f"沿用 {DATA_DIR / NORM_FILE}")
else:
    with (DATA_DIR / "train.csv").open(encoding="utf-8-sig", newline="") as f:
        train_paths = [row["path"] for row in csv.DictReader(f)]
    total, total_sq, count = np.zeros(FEAT_DIM), np.zeros(FEAT_DIM), np.zeros(FEAT_DIM)
    for path in tqdm(train_paths, desc="masked norm"):
        arr = np.load(SEG_DIR / path).astype(np.float64)
        m = valid_mask(arr)
        arr = np.where(m, arr, 0.0)
        total += arr.sum(0)
        total_sq += (arr ** 2).sum(0)
        count += m.sum(0)
    mean = total / np.maximum(count, 1)
    std = np.sqrt(np.maximum(total_sq / np.maximum(count, 1) - mean ** 2, 0)) + 1e-6
    mean[FLAG_DIMS], std[FLAG_DIMS] = 0.0, 1.0
    np.savez(DATA_DIR / NORM_FILE, mean=mean.astype(np.float32), std=std.astype(np.float32),
             valid_ratio=(count / count.max()).astype(np.float32),
             flag_rate=(total / np.maximum(count, 1))[FLAG_DIMS].astype(np.float32))   # 左手、右手、pose、臉的偵測率
    print(f"已存成 {DATA_DIR / NORM_FILE}")

ratio = np.load(DATA_DIR / NORM_FILE)["valid_ratio"]
print(f"訓練集偵測率：左手 {ratio[0]:.1%}、右手 {ratio[63]:.1%}、左腕 {ratio[128 + 3 * 5]:.1%}、右腕 {ratio[128 + 3 * 6]:.1%}、"
      f"臉 {np.load(DATA_DIR / NORM_FILE)['flag_rate'][-1]:.1%}")
print(f"臉部特徵：表情分數 {'開' if USE_BLENDSHAPES else '關'}、嘴唇 {'開' if USE_LIPS else '關'}、頭部轉向 {'開' if USE_HEAD_POSE else '關'}")
with (DATA_DIR / "train.csv").open(encoding="utf-8-sig", newline="") as f:
    first = next(csv.DictReader(f))["path"]
dim = np.load(SEG_DIR / first, mmap_mode="r").shape[1]
assert dim == FEAT_DIM, f"{SEG_DIR / first} 是 {dim} 維，應該是 {FEAT_DIM} 維：請確認 Phase4/5 已經用臉部版本重新產生"

In [ ]:
def load_vocab():
    return json.loads((DATA_DIR / "vocab.json").read_text(encoding="utf-8"))


def augment_geometry(x):
    """隨機旋轉 (pose 與手形一起轉)、pose 縮放與平移。輸入是還沒標準化的原始座標。"""
    x = x.copy()
    T = len(x)
    angle = np.deg2rad(np.random.uniform(-ROTATE_DEG, ROTATE_DEG))
    c, s = np.cos(angle), np.sin(angle)
    rot = np.array([[c, -s], [s, c]], dtype=np.float32)

    pose = x[:, POSE].reshape(T, 9, 3)
    ok = pose[:, :, 2:] > 0
    xy = (pose[:, :, :2] - 0.5) @ rot.T * np.random.uniform(*SCALE_RANGE) + 0.5 \
        + np.random.uniform(-SHIFT, SHIFT, 2).astype(np.float32)
    pose[:, :, :2] = np.where(ok, xy, 0.0)
    x[:, POSE] = pose.reshape(T, -1)

    hands = x[:, :126].reshape(T, 2, 21, 3)          # 手形已經以手腕為原點，直接轉 (沒偵測到的是 0，轉完還是 0)
    hands[..., :2] = hands[..., :2] @ rot.T
    x[:, :126] = hands.reshape(T, -1)
    return x


def time_mask(x):
    """隨機把幾段連續的格整段設成 0 (含旗標)，模擬遮擋或漏偵測。"""
    T = len(x)
    for _ in range(TIME_MASK_N):
        width = np.random.randint(0, max(1, int(T * TIME_MASK_MAX)) + 1)
        if width:
            start = np.random.randint(0, T - width + 1)
            x[start:start + width] = 0.0
    return x


class SignDataset(Dataset):
    def __init__(self, split, vocab, train=False):
        with (DATA_DIR / f"{split}.csv").open(encoding="utf-8-sig", newline="") as f:
            self.rows = [(row["path"], row["text"]) for row in csv.DictReader(f)]
        self.stoi = {c: i for i, c in enumerate(vocab)}
        norm = np.load(DATA_DIR / NORM_FILE)
        self.mean, self.std = norm["mean"], norm["std"]
        self.train = train
        # 快取「原始值」(只做取樣)，資料增強要在標準化之前做
        self.cache = [
            np.ascontiguousarray(np.load(SEG_DIR / path).astype(np.float32)[::FRAME_STRIDE])
            for path, _ in tqdm(self.rows, desc=f"load {split}")
        ]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        text = self.rows[i][1]
        raw = self.cache[i]
        if self.train:
            n = max(1, round(len(raw) * random.uniform(*SPEED_RANGE)))
            raw = raw[np.linspace(0, len(raw) - 1, n).round().astype(int)]
            if AUGMENT:
                raw = augment_geometry(raw)

        mask = valid_mask(raw)
        x = np.where(mask, (raw - self.mean) / self.std, 0.0).astype(np.float32)   # 沒偵測到的一律 0

        if self.train:
            noise = np.random.normal(0, NOISE_STD, x.shape).astype(np.float32)
            noise[:, FLAG_DIMS] = 0.0
            x = x + noise * mask                       # 雜訊只加在有偵測到的座標
            if AUGMENT:
                x = time_mask(x)
        x = x[:MAX_FRAMES]
        ids = [BOS] + [self.stoi.get(c, UNK) for c in text[:MAX_TEXT_LEN - 2]] + [EOS]
        return torch.from_numpy(np.ascontiguousarray(x)), torch.tensor(ids)


def collate(batch):
    xs, ys = zip(*batch)
    src = nn.utils.rnn.pad_sequence(xs, batch_first=True)
    lengths = torch.tensor([len(x) for x in xs])
    src_mask = torch.arange(src.size(1))[None, :] >= lengths[:, None]     # True = padding
    tgt = nn.utils.rnn.pad_sequence(ys, batch_first=True, padding_value=PAD)
    return src, src_mask, tgt

## 篩選詞彙
重新算出上一版的 200 個關鍵詞 (順序跟 `keyword/keywords.json` 相同)，用上一版在**驗證集**的預測算每個詞的 AP，只保留 AP ≥ `SELECT_MIN_VAL_AP` 的詞。

In [ ]:
import logging

import jieba

jieba.setLogLevel(logging.WARNING)
if not JIEBA_DICT.exists():
    try:
        print(f"下載 jieba 繁體詞典到 {JIEBA_DICT} ...")
        urllib.request.urlretrieve(JIEBA_DICT_URL, JIEBA_DICT)
    except Exception as exc:
        print(f"[注意] 下載失敗 ({exc})，改用 jieba 預設詞典；繁體斷詞會比較不準")
if JIEBA_DICT.exists():
    jieba.set_dictionary(str(JIEBA_DICT))
jieba.initialize()

WORD_RE = re.compile(r"[\u4e00-\u9fff]+")
_token_cache = {}


def words_of(text):
    """一句字幕裡的詞 (去重複)，只留中文、長度 >= MIN_WORD_LEN、不在 STOPWORDS 的。"""
    if text not in _token_cache:
        _token_cache[text] = {w for w in jieba.lcut(text)
                              if len(w) >= MIN_WORD_LEN and WORD_RE.fullmatch(w) and w not in STOPWORDS}
    return _token_cache[text]


def read_rows(split):
    with (DATA_DIR / f"{split}.csv").open(encoding="utf-8-sig", newline="") as f:
        return [(row["path"], row["text"]) for row in csv.DictReader(f)]



def keyword_sequence(text):
    """字幕裡依出現順序排列的關鍵詞 (給 CTC 用，重複出現就重複列)。"""
    return [w for w in jieba.lcut(text) if w in KW_INDEX]


def average_precision(y, s):
    """一個詞的 AP：依分數由高到低排序，每找到一個正確的句子就記下當時的精確率，再取平均。"""
    if y.sum() == 0:
        return float("nan")
    order = np.argsort(-s, kind="stable")
    y = y[order]
    precision = np.cumsum(y) / np.arange(1, len(y) + 1)
    return float((precision * y).sum() / y.sum())


# 上一版的 200 個詞 (順序要跟 scores.npz 的欄位一致)
prev_path = PREV_DIR / "keywords.json"
try:
    PREV_KEYWORDS = list(json.loads(prev_path.read_text(encoding="utf-8")))
except Exception:
    raise FileNotFoundError(f"讀不到 {prev_path}：請先跑過 sign_keyword.ipynb")

val_rows = read_rows("val")
prev_val_labels = np.array([[w in words_of(t) for w in PREV_KEYWORDS] for _, t in val_rows], dtype=np.float32)
val_ap = []
for name in PREV_VARIANTS:
    s = np.load(PREV_DIR / name / "scores.npz")["val"]
    assert s.shape[1] == len(PREV_KEYWORDS), "上一版的詞數跟 keywords.json 對不起來"
    val_ap.append([average_precision(prev_val_labels[:, j], s[:, j]) for j in range(len(PREV_KEYWORDS))])
val_ap = np.nanmean(np.array(val_ap, dtype=float), axis=0)

order = [j for j in np.argsort(-np.nan_to_num(val_ap, nan=-1)) if val_ap[j] >= SELECT_MIN_VAL_AP][:SELECT_MAX]
KEYWORDS = [PREV_KEYWORDS[j] for j in order]
KW_INDEX = {w: i for i, w in enumerate(KEYWORDS)}
PREV_COLUMNS = order                              # 上一版 scores 裡對應的欄位，之後比較用

OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "keywords.json").write_text(json.dumps({w: round(float(val_ap[j]), 4) for w, j in zip(KEYWORDS, order)},
                                                  ensure_ascii=False, indent=1), encoding="utf-8")
train_rows = read_rows("train")
covered = sum(bool(words_of(t) & KW_INDEX.keys()) for _, t in train_rows) / len(train_rows)
print(f"上一版 {len(PREV_KEYWORDS)} 個詞 → 保留 {len(KEYWORDS)} 個 (驗證集 AP ≥ {SELECT_MIN_VAL_AP})；"
      f"{covered:.0%} 的訓練句子至少含一個")
print("保留的詞 (依驗證集 AP 排序)：", "、".join(KEYWORDS))
dropped = [PREV_KEYWORDS[j] for j in np.argsort(np.nan_to_num(val_ap, nan=-1))[:20]]
print("刪掉的詞 (AP 最低的 20 個)：", "、".join(dropped))

In [ ]:
class KeywordDataset(SignDataset):
    """輸入跟 SignDataset 相同；答案有兩份：
    labels = 每個關鍵詞有沒有出現 (0/1)；seqs = 關鍵詞依出現順序的編號 (CTC 用，0 留給 blank)。"""

    def __init__(self, split, vocab, train=False):
        super().__init__(split, vocab, train)
        self.labels = np.zeros((len(self.rows), len(KEYWORDS)), dtype=np.float32)
        self.seqs = []
        for i, (_, text) in enumerate(self.rows):
            seq = [KW_INDEX[w] for w in keyword_sequence(text)]
            self.labels[i, seq] = 1.0
            self.seqs.append([j + 1 for j in seq])

    def __getitem__(self, i):
        x, _ = super().__getitem__(i)
        return x, torch.from_numpy(self.labels[i]), torch.tensor(self.seqs[i], dtype=torch.long)


def collate_kw(batch):
    xs, ys, seqs = zip(*batch)
    src = nn.utils.rnn.pad_sequence(xs, batch_first=True)
    lengths = torch.tensor([len(x) for x in xs])
    src_mask = torch.arange(src.size(1))[None, :] >= lengths[:, None]     # True = padding
    targets = torch.cat(seqs) if any(len(s) for s in seqs) else torch.zeros(0, dtype=torch.long)
    return src, src_mask, torch.stack(ys), targets, torch.tensor([len(s) for s in seqs])

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2000):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)]


class SignKeywordCTC(nn.Module):
    """encoder 跟上一版相同。
    分類層：整段特徵 (平均 + 注意力加權) → 每個關鍵詞一個分數
    CTC 層：每一格 → 「blank 或哪個關鍵詞」"""

    def __init__(self, n_keywords, feat_dim=FEAT_DIM):
        super().__init__()
        self.src_proj = nn.Sequential(
            nn.Linear(feat_dim, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(),
            nn.Dropout(DROPOUT), nn.Linear(D_MODEL, D_MODEL),
        )
        self.pos = PositionalEncoding(D_MODEL)
        layer = nn.TransformerEncoderLayer(D_MODEL, N_HEADS, FF_DIM, DROPOUT, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, ENC_LAYERS, norm=nn.LayerNorm(D_MODEL))
        self.attn = nn.Linear(D_MODEL, 1)
        self.head = nn.Sequential(nn.LayerNorm(2 * D_MODEL), nn.Dropout(DROPOUT), nn.Linear(2 * D_MODEL, n_keywords))
        self.ctc_head = nn.Linear(D_MODEL, n_keywords + 1)

    def forward(self, src, src_mask):
        h = self.encoder(self.pos(self.src_proj(src)), src_key_padding_mask=src_mask)
        valid = (~src_mask).unsqueeze(-1).float()
        mean = (h * valid).sum(1) / valid.sum(1).clamp(min=1)
        w = self.attn(h).squeeze(-1).float().masked_fill(src_mask, float("-inf")).softmax(-1)
        attended = (w.unsqueeze(-1) * h.float()).sum(1)
        logits = self.head(torch.cat([mean.float(), attended], dim=-1))
        return logits, self.ctc_head(h).float()

## 評估函式
跟上一版相同，另外加上「CTC 偵測分數」：每個詞在整句中 CTC 機率的最大值。

In [ ]:
def average_precision(y, s):
    """一個詞的 AP：依分數由高到低排序，每找到一個正確的句子就記下當時的精確率，再取平均。"""
    if y.sum() == 0:
        return float("nan")
    order = np.argsort(-s, kind="stable")
    y = y[order]
    precision = np.cumsum(y) / np.arange(1, len(y) + 1)
    return float((precision * y).sum() / y.sum())


def per_word_ap(labels, scores):
    return np.array([average_precision(labels[:, j], scores[:, j]) for j in range(labels.shape[1])])


def eval_words(labels):
    """測試集出現至少 MIN_TEST_POSITIVE 次的詞才算進 mAP。"""
    return labels.sum(0) >= MIN_TEST_POSITIVE


def mean_ap(labels, scores, mask=None):
    ap = per_word_ap(labels, scores)
    mask = eval_words(labels) if mask is None else mask
    return float(np.nanmean(ap[mask]))


def shuffle_within(scores, groups, seed=SEED):
    """把預測在同一組 (同一則新聞) 的句子之間打亂。"""
    rng = np.random.default_rng(seed)
    out = scores.copy()
    by_group = defaultdict(list)
    for i, g in enumerate(groups):
        by_group[g].append(i)
    for idx in by_group.values():
        out[idx] = scores[rng.permutation(idx)]
    return out


def best_thresholds(labels, scores):
    """每個詞在驗證集上挑一個讓 F1 最高的門檻。"""
    out = np.full(labels.shape[1], 0.5)
    for j in range(labels.shape[1]):
        y, s = labels[:, j], scores[:, j]
        if y.sum() == 0:
            continue
        order = np.argsort(-s)
        tp = np.cumsum(y[order])
        f1 = 2 * tp / (np.arange(1, len(y) + 1) + y.sum())
        k = int(f1.argmax())
        out[j] = s[order[k]]
    return out


def micro_f1(labels, pred):
    tp = (labels * pred).sum()
    return float(2 * tp / max(1.0, labels.sum() + pred.sum()))


@torch.no_grad()
def predict_scores(model, loader, keep_ctc=False):
    """回傳 (分類層機率, CTC 偵測分數[, 每一格的 CTC 機率])。"""
    model.eval()
    cls, det, frames = [], [], []
    for src, src_mask, *_ in loader:
        src, src_mask = src.to(DEVICE), src_mask.to(DEVICE)
        with torch.autocast(DEVICE, enabled=USE_AMP):
            logits, ctc_logits = model(src, src_mask)
        prob = ctc_logits.softmax(-1).masked_fill(src_mask.unsqueeze(-1), 0.0)
        cls.append(torch.sigmoid(logits.float()).cpu().numpy())
        det.append(prob[:, :, 1:].amax(1).cpu().numpy())
        if keep_ctc:
            lengths = (~src_mask).sum(1).tolist()
            frames += [p[:n].cpu().numpy() for p, n in zip(prob, lengths)]
    return (np.concatenate(cls), np.concatenate(det)) + ((frames,) if keep_ctc else ())

## 訓練兩個版本
結果存在 `keyword_ctc/<版本名稱>/`：`scores.npz` (驗證集、測試集的分類層機率與 CTC 偵測分數)、`run_info.json`、`best.pt`。中途中斷重新執行，已完成的版本會自動略過。

In [ ]:
vocab = load_vocab()
train_set = KeywordDataset("train", vocab, train=True)
val_set = KeywordDataset("val", vocab)
test_set = KeywordDataset("test", vocab)
val_loader = DataLoader(val_set, BATCH_SIZE, collate_fn=collate_kw)
test_loader = DataLoader(test_set, BATCH_SIZE, collate_fn=collate_kw)
print(f"每句平均含 {train_set.labels.sum(1).mean():.2f} 個關鍵詞；"
      f"測試集有 {int(eval_words(test_set.labels).sum())} 個詞出現 >= {MIN_TEST_POSITIVE} 次 (算進 mAP)")

USE_BLENDSHAPES = USE_LIPS = USE_HEAD_POSE = USE_FACE
print("臉部特徵：", "使用" if USE_FACE else "不使用 (全部遮掉)")


def train_variant(name, ctc_weight):
    out = OUT_DIR / name
    if (out / "scores.npz").exists() and (out / "run_info.json").exists():
        print(f"{name}：已經跑過，沿用 {out}")
        return
    out.mkdir(parents=True, exist_ok=True)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    train_loader = DataLoader(train_set, BATCH_SIZE, shuffle=True, collate_fn=collate_kw)
    model = SignKeywordCTC(len(KEYWORDS)).to(DEVICE)
    pos = train_set.labels.sum(0)
    pos_weight = torch.tensor(np.clip(np.sqrt((len(train_set) - pos) / np.maximum(pos, 1)), 1, MAX_POS_WEIGHT),
                              dtype=torch.float32, device=DEVICE)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    ctc = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps, warmup = EPOCHS * len(train_loader), WARMUP_EPOCHS * len(train_loader)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda step: step / warmup if step < warmup
        else 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total_steps - warmup))),
    )
    scaler = torch.amp.GradScaler(enabled=USE_AMP)

    history, best_map, best_epoch, stale = [], -1.0, 0, 0
    bar = tqdm(range(1, EPOCHS + 1), desc=name)
    for epoch in bar:
        model.train()
        total, n = 0.0, 0
        for src, src_mask, y, targets, target_lengths in train_loader:
            src, src_mask, y = src.to(DEVICE), src_mask.to(DEVICE), y.to(DEVICE)
            with torch.autocast(DEVICE, enabled=USE_AMP):
                logits, ctc_logits = model(src, src_mask)
                loss = bce(logits.float(), y)
                if ctc_weight > 0:
                    log_probs = ctc_logits.log_softmax(-1).transpose(0, 1)        # (T, N, C)
                    loss = loss + ctc_weight * ctc(log_probs, targets.to(DEVICE), (~src_mask).sum(1),
                                                   target_lengths.to(DEVICE))
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total, n = total + loss.item() * len(y), n + len(y)

        val_cls, _ = predict_scores(model, val_loader)
        val_map = mean_ap(val_set.labels, val_cls)          # 兩個版本都用分類層的 val mAP 選模型
        history.append((total / n, val_map))
        if val_map > best_map:
            best_map, best_epoch, stale = val_map, epoch, 0
            torch.save(model.state_dict(), out / "best.pt")
        else:
            stale += 1
        bar.set_postfix(loss=f"{total / n:.4f}", val_mAP=f"{val_map:.3f}", best=f"{best_map:.3f}")
        if stale >= PATIENCE:
            break

    model.load_state_dict(torch.load(out / "best.pt", map_location=DEVICE))
    val_cls, val_det = predict_scores(model, val_loader)
    test_cls, test_det = predict_scores(model, test_loader)
    np.savez(out / "scores.npz", val=val_cls, test=test_cls, val_ctc=val_det, test_ctc=test_det)
    (out / "run_info.json").write_text(json.dumps(
        {"ctc_weight": ctc_weight, "best_epoch": best_epoch, "best_val_map": best_map, "history": history},
        ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"{name} 完成：最佳 epoch {best_epoch}，val mAP {best_map:.3f}")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


for name, ctc_weight in VARIANTS.items():
    train_variant(name, ctc_weight)

## 結果
- **上一版**：上一版模型 (200 個詞) 的預測，只取這次保留的詞重新計算，讓比較公平
- **CTC 偵測**：不用分類層，直接用 CTC 每個詞在整句中的最大機率當分數
- **CTC 平均**：分類層與 CTC 偵測分數取平均

In [ ]:
labels, val_labels = test_set.labels, val_set.labels
clips = [p.rsplit("_s", 1)[0] for p, _ in test_set.rows]
mask = eval_words(labels)

rows = {}
for name in PREV_VARIANTS:                               # 上一版，只取保留的詞
    s = np.load(PREV_DIR / name / "scores.npz")
    rows[f"上一版 {name}"] = (s["val"][:, PREV_COLUMNS], s["test"][:, PREV_COLUMNS])
for name, w in VARIANTS.items():
    s = np.load(OUT_DIR / name / "scores.npz")
    rows[name] = (s["val"], s["test"])
    if w > 0:
        rows[f"{name} (CTC 偵測)"] = (s["val_ctc"], s["test_ctc"])
        rows[f"{name} (平均)"] = ((s["val"] + s["val_ctc"]) / 2, (s["test"] + s["test_ctc"]) / 2)

results = {}
for name, (val_s, test_s) in rows.items():
    thr = best_thresholds(val_labels, val_s)
    results[name] = {
        "scores": test_s, "ap": per_word_ap(labels, test_s),
        "mAP": mean_ap(labels, test_s, mask),
        "micro_AP": average_precision(labels.ravel(), test_s.ravel()),
        "F1": micro_f1(labels, test_s >= thr),
        "mAP_clip_shuffle": mean_ap(labels, shuffle_within(test_s, clips), mask),
    }

prior = float(labels[:, mask].mean(0).mean())
print(f"測試集 {len(labels)} 句，{int(mask.sum())} 個詞算進 mAP\n")
print(f"{'版本':<20}{'mAP':>8}{'micro AP':>10}{'F1':>8}{'同則打亂mAP':>14}")
for name, r in results.items():
    print(f"{name:<20}{r['mAP']:>8.3f}{r['micro_AP']:>10.3f}{r['F1']:>8.3f}{r['mAP_clip_shuffle']:>15.3f}")
print(f"{'亂猜':<20}{prior:>8.3f}")


def bootstrap_diff(a, b, n=1000, seed=SEED):
    """以新聞為單位重抽，回傳 mAP(a) − mAP(b) 的中位數與 95% 區間。"""
    rng = np.random.default_rng(seed)
    by_clip = defaultdict(list)
    for i, c in enumerate(clips):
        by_clip[c].append(i)
    groups = list(by_clip.values())
    diffs = []
    for _ in range(n):
        idx = np.concatenate([groups[k] for k in rng.integers(0, len(groups), len(groups))])
        y = labels[idx]
        m = eval_words(y)
        diffs.append(mean_ap(y, results[a]["scores"][idx], m) - mean_ap(y, results[b]["scores"][idx], m))
    d = np.sort(diffs)
    return d[n // 2], d[int(n * 0.025)], d[int(n * 0.975)]


print()
for a, b in (("無CTC", f"上一版 {PREV_VARIANTS[-1]}"), ("CTC", "無CTC"), ("CTC (平均)", "無CTC")):
    if a in results and b in results:
        med, lo, hi = bootstrap_diff(a, b)
        verdict = "差距確實存在" if lo > 0 or hi < 0 else "區間包含 0，可能只是隨機誤差"
        print(f"mAP 差距 ({a} − {b})：{med:+.3f}，95% 區間 [{lo:+.3f}, {hi:+.3f}] → {verdict}")

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "Noto Sans CJK TC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

compare = [n for n in (f"上一版 {PREV_VARIANTS[-1]}", "無CTC", "CTC", "CTC (平均)") if n in results]
main = compare[-1]
order = [j for j in np.argsort(-np.nan_to_num(results[main]["ap"], nan=-1)) if mask[j]][:40]

fig, ax = plt.subplots(figsize=(16, 9))
y = np.arange(len(order))
height = 0.8 / len(compare)
for k, name in enumerate(compare):
    ax.barh(y + k * height, results[name]["ap"][order], height=height, label=name)
ax.set_yticks(y + 0.4 - height / 2, [f"{KEYWORDS[j]} ({int(labels[:, j].sum())})" for j in order])
ax.invert_yaxis()
ax.set_xlabel("AP")
ax.set_title(f"AP 最高的 40 個詞 (依「{main}」排序；括號 = 測試集出現次數)")
ax.legend()
plt.tight_layout()
plt.show()

## 詞的時間位置：模型認為每個詞出現在第幾秒
用 `CTC` 版本，畫出每一格「是某個關鍵詞」的機率 (只畫機率最高的幾個詞)。
`.npy` 沒有被拆開，位置是模型從整句中自己學出來的；字幕本身沒有詞的時間標記，所以這只能用眼睛確認是否合理 (例如對照影片)。

In [ ]:
CTC_NAME = next((n for n, w in VARIANTS.items() if w > 0), None)
SHOW = 6            # 畫幾句
LINES = 4           # 每句畫幾個詞

if CTC_NAME is None:
    print("VARIANTS 裡沒有使用 CTC 的版本")
else:
    model = SignKeywordCTC(len(KEYWORDS)).to(DEVICE)
    model.load_state_dict(torch.load(OUT_DIR / CTC_NAME / "best.pt", map_location=DEVICE))
    _, _, frames = predict_scores(model, test_loader, keep_ctc=True)

    rng = np.random.default_rng(1)
    picks = [i for i in rng.permutation(len(labels)) if labels[i].sum() > 0 and frames[i][:, 1:].max() > 0.3][:SHOW]
    fig, axes = plt.subplots(len(picks), 1, figsize=(14, 2.6 * len(picks)), squeeze=False)
    for ax, i in zip(axes[:, 0], picks):
        path, text = test_set.rows[i]
        prob = frames[i]                                       # (格數, 1 + 詞數)，第 0 欄是 blank
        t = np.arange(len(prob)) * FRAME_STRIDE / FPS          # 換成原始影片的秒數
        top = np.argsort(-prob[:, 1:].max(0))[:LINES]
        truth = set(np.flatnonzero(labels[i]))
        for j in top:
            ax.plot(t, prob[:, j + 1], label=f"{'✓' if j in truth else '✗'} {KEYWORDS[j]}")
        ax.set_ylim(0, 1.05)
        ax.set_title(f"{text}　(字幕裡的關鍵詞：{'、'.join(KEYWORDS[j] for j in sorted(truth))})", fontsize=10)
        ax.legend(loc="upper right", fontsize=9)
    axes[-1, 0].set_xlabel("秒 (從這句開頭算起)")
    plt.tight_layout()
    plt.show()

    print("文字版：CTC 解碼出的詞與時間位置")
    for i in picks:
        prob = frames[i]
        best = prob.argmax(1)
        spans, start = [], None
        for k in range(len(best) + 1):
            cur = best[k] if k < len(best) else 0
            if start is not None and cur != best[start]:
                spans.append((KEYWORDS[best[start] - 1], start, k - 1))
                start = None
            if start is None and cur != 0:
                start = k
        text = test_set.rows[i][1]
        found = "、".join(f"{w} {s * FRAME_STRIDE / FPS:.1f}~{(e + 1) * FRAME_STRIDE / FPS:.1f} 秒" for w, s, e in spans) or "(沒有偵測到)"
        print(f"  {text}\n    → {found}")

## 怎麼解讀

- **無CTC vs 上一版**：只篩選詞彙有沒有幫助 (兩者算的是同一批詞)
- **CTC vs 無CTC**：CTC 輔助損失有沒有幫助；bootstrap 區間不包含 0 才算確實有差
- **CTC (偵測) / (平均)**：如果 CTC 偵測分數本身也很高，代表模型確實能在時間上找到這些詞
- **時間位置圖**：✓ 的詞在某個時間點出現明顯尖峰，代表模型認為這個詞的手勢在那裡；之後可以用這些位置從整句 `.npy` 切出單詞片段 (另外的步驟，不影響 Phase5 的切點)